# SENTINEL: data quality, feature engineering and reason codes
Steps 1 to 5 of the assignment on the SENTINEL fraud data (Project Work 4 and 5).
1. Load customers, merchants, devices, transactions, payment_outcomes, labels (+ features to compare).
2. Data quality report.
3. Compute six features using only data from before each transaction.
4. Assign reason codes R01 to R08 with the thresholds from class.
5. Compare with the provided features table and explain every mismatch.

Data source: the SENTINEL tables folder (default) or your own Supabase load (set SOURCE=supabase in .env).

## Step 0: Settings

In [1]:
import os                       # read settings from environment variables
import glob                     # find the part-*.csv.gz files inside the tables folder
import json                     # parse the JSON location columns and reason code lists
import zipfile                  # read the tables straight out of the SENTINEL zip, if you point to the zip

import numpy as np              # fast maths (haversine, comparisons)
import pandas as pd             # all the table work
import requests                 # only used when SOURCE=supabase (REST API calls)
from dotenv import load_dotenv  # loads settings from a .env file next to this script

load_dotenv()                                   # read .env if it exists (no error if it does not)
pd.set_option("display.width", 200)             # wider printouts so tables are not wrapped
pd.set_option("display.max_columns", 30)        # show up to 30 columns when printing a DataFrame

SOURCE = os.getenv("SOURCE", "files").lower()   # "files" (tables folder or zip) or "supabase"
DATA_PATH = os.getenv(                          # where the SENTINEL tables are on your laptop
    "DATA_PATH",                                # name of the setting in .env
    r"C:\Users\sudha\Downloads\Sentinel project data\sentinel_data\tables",  # default: your extracted tables folder
)
SAMPLE_CUSTOMERS = int(os.getenv("SAMPLE_CUSTOMERS", "0"))  # 0 = all 25,000 customers; e.g. 2000 = faster run
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "outputs")             # folder where the result CSV files are written
os.makedirs(OUTPUT_DIR, exist_ok=True)                      # create the output folder if it is missing

TABLES = ["customers", "merchants", "devices", "transactions", "payment_outcomes", "labels", "features"]  # tables we need
PRIMARY_KEYS = {                     # the primary key of each table (used for the duplicate check)
    "customers": "customer_id",      # one row per customer
    "merchants": "merchant_id",      # one row per merchant
    "devices": "device_id",          # one row per device
    "transactions": "transaction_id",      # one row per payment attempt
    "payment_outcomes": "transaction_id",  # one outcome per transaction
    "labels": "transaction_id",            # one fraud label per transaction
    "features": "transaction_id",          # one feature row per transaction
}

# Thresholds from class (Project Work 4 transcript, reason code walkthrough at 00:38:00)
NEW_DEVICE_DAYS = 90            # R01: device not seen for this customer in the preceding 90 days
GEO_DISTANCE_KM = 500           # R02: customer to merchant distance above 500 km (or a new city)
NIGHT_START, NIGHT_END = 0, 5   # R03: night = 00:00 (inclusive) to 05:00 (exclusive) IST
FAILED_MIN = 3                  # R04: at least 3 failed attempts in the preceding 24 hours
SPIKE_RATIO = 5                 # R05: amount at least 5 times the 90-day average
NEW_ACCOUNT_DAYS = 30           # R06: account younger than 30 days
VELOCITY_MIN = 4                # R07: at least 4 preceding attempts within 1 hour
HIGH_RISK_MCC = {"gaming", "gift_cards", "electronics"}  # R08: high-risk merchant categories
EARTH_RADIUS_KM = 6371.0        # mean earth radius used by the haversine formula
IST = "Asia/Kolkata"            # SENTINEL stores UTC; the business rules use Indian time (UTC+5:30)

## Step 1: Load the tables into pandas
Two ways to load:
- **files**: the `part-*.csv.gz` files in the tables folder (or inside the zip). Only `.csv.gz` files are read,
  because the folder also has a few extracted `.csv` copies that would load the same rows twice.
- **supabase**: your own Supabase load from Project Work 5 (`sql/create_table` + `python ingest.py all`),
  read with the REST API, 1,000 rows per request.

In [2]:
def load_from_files(path):
    """Read every table from the tables folder (or the zip) into a dict of DataFrames."""
    frames = {}                                          # table name -> DataFrame
    if path.lower().endswith(".zip"):                    # the user pointed to the zip file
        z = zipfile.ZipFile(path)                        # open the zip without extracting it
        names = z.namelist()                             # every file path inside the zip
        for t in TABLES:                                 # go through the tables we need
            parts = sorted(n for n in names              # pick this table's part files ...
                           if f"/tables/{t}/" in n and n.endswith(".csv.gz"))  # ... only the .csv.gz ones
            frames[t] = pd.concat(                       # stack all parts into one DataFrame
                [pd.read_csv(z.open(p), compression="gzip") for p in parts],  # read each gzip CSV
                ignore_index=True)                       # give the result a fresh 0..n-1 index
    else:                                                # the user pointed to the extracted tables folder
        for t in TABLES:                                 # go through the tables we need
            parts = sorted(glob.glob(os.path.join(path, t, "*", "part-*.csv.gz")))  # e.g. transactions/train/part-00001.csv.gz
            frames[t] = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)  # read and stack
    return frames                                        # dict of all tables


def load_from_supabase(sample_customers):
    """Read the tables from your own Supabase project with the REST API."""
    base = os.environ["SUPABASE_URL"].rstrip("/") + "/rest/v1/"   # e.g. https://xxxx.supabase.co/rest/v1/
    key = os.environ["SUPABASE_KEY"]                               # your anon/service key from .env
    schema = os.getenv("SUPABASE_SCHEMA", "public")                # schema the tables live in
    headers = {"apikey": key, "Authorization": f"Bearer {key}"}    # Supabase needs the key in both headers
    if schema != "public":                                         # non-public schema (e.g. sentinel_demo) ...
        headers["Accept-Profile"] = schema                         # ... must be named in this header

    def get(table, params):                                        # one GET request, returns a list of rows
        r = requests.get(base + table, headers=headers, params=params, timeout=120)  # call the REST API
        r.raise_for_status()                                       # stop with a clear error if URL/key/table is wrong
        return r.json()                                            # rows as a list of dicts

    def read_all(table, order):                                    # read a whole table, 1,000 rows at a time
        rows, offset = [], 0                                       # collected rows and current position
        while True:                                                # keep paging until a page comes back short
            page = get(table, {"select": "*", "order": order, "limit": 1000, "offset": offset})  # next page
            rows += page                                           # add the page to the result
            offset += len(page)                                    # move the position forward
            if len(page) < 1000:                                   # a short page means we reached the end
                break                                              # stop paging
        print(f"  {table}: {len(rows):,} rows")                    # progress message
        return pd.DataFrame(rows)                                  # rows -> DataFrame

    def read_in(table, column, ids):                               # read only rows whose column is in ids
        out = []                                                   # collected DataFrames
        ids = list(dict.fromkeys(ids))                             # unique ids, keep order
        for i in range(0, len(ids), 150):                          # 150 ids per request keeps the URL short
            chunk = ",".join(ids[i:i + 150])                       # comma separated id list
            offset = 0                                             # Supabase returns at most 1,000 rows per request ...
            while True:                                            # ... so page through each id chunk too
                page = get(table, {"select": "*", column: f"in.({chunk})", "order": "transaction_id",
                                   "limit": 1000, "offset": offset})  # next page of the filtered rows
                out.append(pd.DataFrame(page))                     # keep the page
                offset += len(page)                                # move the position forward
                if len(page) < 1000:                               # a short page means this chunk is done
                    break                                          # go to the next chunk
        df = pd.concat(out, ignore_index=True)                     # stack all chunks
        print(f"  {table}: {len(df):,} rows")                      # progress message
        return df                                                  # filtered table

    frames = {}                                                    # table name -> DataFrame
    frames["customers"] = read_all("customers", "customer_id")     # 25,000 rows, small enough to read fully
    frames["merchants"] = read_all("merchants", "merchant_id")     # 2,000 rows
    frames["devices"] = read_all("devices", "device_id")           # about 100,000 rows
    if sample_customers:                                           # sample mode: a few customers, full history each
        ids = frames["customers"].customer_id.sample(sample_customers, random_state=42).tolist()  # pick customers
        frames["transactions"] = read_in("transactions", "customer_id", ids)  # every transaction of those customers
        tx_ids = frames["transactions"].transaction_id.tolist()               # their transaction ids
        for t in ["payment_outcomes", "labels", "features"]:                  # the 1:1 tables
            frames[t] = read_in(t, "transaction_id", tx_ids)                  # only rows of those transactions
    else:                                                          # full mode: read every row
        for t in ["transactions", "payment_outcomes", "labels", "features"]:  # the big tables
            frames[t] = read_all(t, "transaction_id")                         # page through all rows
    for df in frames.values():                                     # Supabase returns jsonb as dicts/lists ...
        for col in df.columns:                                     # ... in any table (e.g. merchants.merchant_location)
            if df[col].map(lambda v: isinstance(v, (dict, list))).any():  # a jsonb column
                df[col] = df[col].map(lambda v: json.dumps(v) if isinstance(v, (dict, list)) else v)  # -> JSON text, same as the CSV files
    return frames                                                  # dict of all tables


print(f"Loading from: {SOURCE}")                                   # say which source we use
data = load_from_supabase(SAMPLE_CUSTOMERS) if SOURCE == "supabase" else load_from_files(DATA_PATH)  # load

if SOURCE != "supabase" and SAMPLE_CUSTOMERS:                      # sample mode for the files source
    keep = data["customers"].customer_id.sample(SAMPLE_CUSTOMERS, random_state=42)  # pick random customers
    data["transactions"] = data["transactions"][data["transactions"].customer_id.isin(keep)]  # keep ALL their transactions (full history)
    tx_ids = set(data["transactions"].transaction_id)              # transaction ids we kept
    for t in ["payment_outcomes", "labels", "features"]:           # the 1:1 tables
        data[t] = data[t][data[t].transaction_id.isin(tx_ids)]     # keep only matching rows

customers, merchants, devices = data["customers"], data["merchants"], data["devices"]       # short names
transactions, outcomes, labels = data["transactions"], data["payment_outcomes"], data["labels"]  # short names
features_given = data["features"]                                  # the provided features table (for step 5)

for name, df in data.items():                                      # show the size of every table
    print(f"{name:17s} {len(df):>10,} rows x {df.shape[1]} columns")  # e.g. transactions 1,100,000 rows x 18 columns

Loading from: files


customers             25,000 rows x 6 columns
merchants              2,000 rows x 5 columns
devices              100,120 rows x 3 columns
transactions       1,100,000 rows x 18 columns
payment_outcomes   1,100,000 rows x 5 columns
labels             1,100,000 rows x 7 columns
features           1,100,000 rows x 22 columns


## Step 2: Data quality report
For every table: nulls per column, duplicate keys, duplicate rows, values with the wrong type.
For transactions: outliers in the amount. For the split tables: row count per split.

In [3]:
# What type each column should have (from sql/schema.sql). Anything that does not parse is a "wrong type".
EXPECTED_TYPES = {
    "customers": {"created_at": "timestamp", "customer_age": "int", "is_synthetic": "bool"},
    "merchants": {"merchant_location": "json", "is_synthetic": "bool"},
    "devices": {"is_synthetic": "bool"},
    "transactions": {"amount_paise": "int", "transaction_amount": "float", "transaction_time": "timestamp",
                     "customer_age": "int", "customer_location": "json", "merchant_location": "json",
                     "account_age": "int", "previous_transactions": "int",
                     "average_transaction_value": "float", "failed_transactions": "int", "is_synthetic": "bool"},
    "payment_outcomes": {"outcome_time": "timestamp", "is_synthetic": "bool"},
    "labels": {"fraud_label": "int", "label_available_at": "timestamp",
               "eligible_at_next_split_start": "bool", "is_synthetic": "bool"},
    "features": {"feature_as_of": "timestamp", "amount_to_avg_ratio": "float", "is_new_device": "bool",
                 "customer_merchant_distance_km": "float", "hour_of_day": "int", "is_night": "bool",
                 "txn_count_last_1h": "int", "txn_amount_last_24h_paise": "int", "reason_codes": "json"},
}


def count_wrong_type(series, kind):
    """Count non-null values that do not parse as the expected type."""
    s = series.dropna()                                             # nulls are counted separately, skip them
    if kind == "timestamp":                                         # should be an ISO-8601 time
        return int(pd.to_datetime(s, utc=True, format="ISO8601", errors="coerce").isna().sum())  # values that fail to parse
    if kind in ("int", "float"):                                    # should be a number
        num = pd.to_numeric(s, errors="coerce")                     # text that is not a number becomes NaN
        bad = num.isna()                                            # not a number at all
        if kind == "int":                                           # integers must also have no fraction
            bad |= (num.notna() & (num % 1 != 0))                   # e.g. 3.5 in an integer column
        return int(bad.sum())                                       # number of bad values
    if kind == "bool":                                              # should be True/False
        return int((~s.astype(str).str.lower().isin(["true", "false", "1", "0"])).sum())  # anything else is wrong
    if kind == "json":                                              # should be valid JSON text
        def bad_json(v):                                            # True when v is not valid JSON
            if isinstance(v, (dict, list)):                         # already parsed JSON (e.g. from the database)
                return False                                        # that is valid
            try:                                                    # try to parse it
                json.loads(v)                                       # parse the text
                return False                                        # it parsed fine
            except (TypeError, ValueError):                         # it did not parse
                return True                                         # count it as wrong
        return int(s.map(bad_json).sum())                           # number of bad values
    return 0                                                        # no check defined for this kind


# Supabase jsonb columns can arrive as dicts/lists, which pandas cannot compare: turn them into JSON text
for df in data.values():                                            # every loaded table
    for col in df.columns:                                          # every column
        if df[col].map(lambda v: isinstance(v, (dict, list))).any():  # a column holding dicts or lists
            df[col] = df[col].map(lambda v: json.dumps(v) if isinstance(v, (dict, list)) else v)  # -> JSON text

dq_rows = []                                                        # one row per table and column
for name, df in data.items():                                       # every loaded table
    pk = PRIMARY_KEYS[name]                                         # its primary key column
    dup_keys = int(df[pk].duplicated().sum())                       # rows whose key appeared before
    dup_rows = int(df.duplicated().sum())                           # rows that are an exact copy of another row
    for col in df.columns:                                          # every column of the table
        kind = EXPECTED_TYPES.get(name, {}).get(col, "text")        # expected type, text if not listed
        dq_rows.append({                                            # one line of the report
            "table": name,                                          # table name
            "column": col,                                          # column name
            "rows": len(df),                                        # rows in the table
            "nulls": int(df[col].isna().sum()),                     # empty values
            "null_%": round(100 * df[col].isna().mean(), 2),        # empty values as a percentage
            "expected_type": kind,                                  # what the type should be
            "wrong_type": count_wrong_type(df[col], kind),          # values that do not parse
            "duplicate_keys_in_table": dup_keys,                    # same for every column of the table
            "duplicate_rows_in_table": dup_rows,                    # same for every column of the table
        })
dq = pd.DataFrame(dq_rows)                                          # report as a DataFrame
dq.to_csv(f"{OUTPUT_DIR}/step2_quality_by_column.csv", index=False)  # save it

summary = (dq.groupby("table")                                      # one line per table
             .agg(rows=("rows", "first"),                           # row count
                  columns=("column", "count"),                      # column count
                  columns_with_nulls=("nulls", lambda x: int((x > 0).sum())),  # how many columns have nulls
                  total_nulls=("nulls", "sum"),                     # all nulls in the table
                  wrong_type_values=("wrong_type", "sum"),          # all wrong-type values
                  duplicate_keys=("duplicate_keys_in_table", "first"),   # duplicate primary keys
                  duplicate_rows=("duplicate_rows_in_table", "first")))  # exact duplicate rows
print(summary)                                                      # print the per-table summary
print("\nColumns with nulls:")                                      # heading
print(dq[dq.nulls > 0][["table", "column", "nulls", "null_%"]].to_string(index=False))  # only columns with nulls

# Extra checks a type test cannot see
extra = {                                                           # name of check -> number of bad rows
    "amount_paise <= 0": int((transactions.amount_paise <= 0).sum()),       # amounts must be positive
    "amount_paise != transaction_amount x 100": int(                        # the two amount columns must agree
        (transactions.amount_paise != (transactions.transaction_amount * 100).round().astype("int64")).sum()),
    "customer_age outside 18-80": int((~transactions.customer_age.between(18, 80)).sum()),  # KYC age range
    "transactions with unknown customer_id": int((~transactions.customer_id.isin(customers.customer_id)).sum()),  # orphan rows
    "transactions with unknown merchant_id": int((~transactions.merchant_id.isin(merchants.merchant_id)).sum()),  # orphan rows
    "transactions with unknown device_id": int((transactions.device_id.notna()                                      # device given ...
                                                 & ~transactions.device_id.isin(devices.device_id)).sum()),         # ... but not in devices
    "transactions without a label": int((~transactions.transaction_id.isin(labels.transaction_id)).sum()),        # every tx needs a label
    "transactions without an outcome": int((~transactions.transaction_id.isin(outcomes.transaction_id)).sum()),   # and an outcome
    "FAILED outcome without failure_code": int(((outcomes.status == "FAILED") & outcomes.failure_code.isna()).sum()),  # consistency
}
extra = pd.Series(extra, name="bad_rows")                           # turn into a Series for printing
print("\nExtra checks:")                                            # heading
print(extra.to_string())                                            # print each check and its count
extra.to_csv(f"{OUTPUT_DIR}/step2_extra_checks.csv")                # save it

# Outliers in amount: Tukey rule (1.5 x IQR) on the amount and on log10(amount)
amt = transactions.transaction_amount                               # amount in rupees
q1, q3 = amt.quantile([0.25, 0.75])                                 # first and third quartile
iqr = q3 - q1                                                       # interquartile range
upper = q3 + 1.5 * iqr                                              # values above this are outliers (plain scale)
log_amt = np.log10(amt)                                             # amounts are very skewed, so also check the log scale
lq1, lq3 = log_amt.quantile([0.25, 0.75])                           # quartiles of the log amount
log_upper = 10 ** (lq3 + 1.5 * (lq3 - lq1))                         # upper fence turned back into rupees
log_lower = 10 ** (lq1 - 1.5 * (lq3 - lq1))                         # lower fence turned back into rupees
outliers = pd.Series({                                              # summary of the outlier check
    "min (Rs)": amt.min(),                                          # smallest amount
    "median (Rs)": amt.median(),                                    # typical amount
    "max (Rs)": amt.max(),                                          # largest amount
    "IQR upper fence (Rs)": round(upper, 2),                        # plain Tukey fence
    "rows above IQR fence": int((amt > upper).sum()),               # outliers on the plain scale
    "log-IQR lower fence (Rs)": round(log_lower, 2),                # Tukey fence on log scale (low side)
    "log-IQR upper fence (Rs)": round(log_upper, 2),                # Tukey fence on log scale (high side)
    "rows outside log-IQR fences": int(((amt > log_upper) | (amt < log_lower)).sum()),  # outliers on the log scale
    "rows above Rs 10 lakh": int((amt > 1_000_000).sum()),          # very large single payments
})
print("\nAmount outliers:")                                         # heading
print(outliers.to_string())                                         # print the summary
outliers.to_csv(f"{OUTPUT_DIR}/step2_amount_outliers.csv")          # save it
print(transactions.nlargest(5, "transaction_amount")[["transaction_id", "customer_id", "transaction_amount", "merchant_category", "split"]])  # the 5 biggest payments

# Transactions per IST hour of day (checks the time column and the night window)
hours_ist = pd.to_datetime(transactions.transaction_time, utc=True, format="ISO8601").dt.tz_convert(IST).dt.hour  # UTC -> IST hour
per_hour = hours_ist.value_counts().reindex(range(24), fill_value=0)  # rows per hour 0..23, missing hours shown as 0
print("\nTransactions per IST hour:")                                # heading
print(per_hour.to_string())                                         # print the counts
print("Empty hours:", list(per_hour[per_hour == 0].index))          # hours with no transactions at all

# Row count per split
split_counts = pd.DataFrame({                                       # one column per table that has a split
    "transactions": transactions.split.value_counts(),              # rows per split in transactions
    "labels": labels.split.value_counts(),                          # rows per split in labels
    "fraud_rate_%": (labels.groupby("split").fraud_label.mean() * 100).round(3),  # fraud share per split
}).reindex(["warmup", "train", "validation", "holdout"])            # fixed, time-ordered order
split_counts.loc["total"] = split_counts.sum(numeric_only=True)     # add a total row
split_counts.loc["total", "fraud_rate_%"] = round(labels.fraud_label.mean() * 100, 3)  # overall fraud rate, not a sum
print("\nRows per split:")                                          # heading
print(split_counts)                                                 # print the table
split_counts.to_csv(f"{OUTPUT_DIR}/step2_rows_per_split.csv")       # save it

                     rows  columns  columns_with_nulls  total_nulls  wrong_type_values  duplicate_keys  duplicate_rows
table                                                                                                                 
customers           25000        6                   0            0                  0               0               0
devices            100120        3                   0            0                  0               0               0
features          1100000       22                   5       115825                  0               0               0
labels            1100000        7                   0            0                  0               0               0
merchants            2000        5                   0            0                  0               0               0
payment_outcomes  1100000        5                   1       981435                  0               0               0
transactions      1100000       18              


Extra checks:
amount_paise <= 0                           0
amount_paise != transaction_amount x 100    0
customer_age outside 18-80                  0
transactions with unknown customer_id       0
transactions with unknown merchant_id       0
transactions with unknown device_id         0
transactions without a label                0
transactions without an outcome             0
FAILED outcome without failure_code         0

Amount outliers:
min (Rs)                             2.17
median (Rs)                       1250.64
max (Rs)                       7987337.17
IQR upper fence (Rs)              5966.58
rows above IQR fence            102543.00
log-IQR lower fence (Rs)            56.16
log-IQR upper fence (Rs)         28121.49
rows outside log-IQR fences      13000.00
rows above Rs 10 lakh              349.00
       transaction_id         customer_id  transaction_amount merchant_category    split
176736  TX_0000136285  C_fb660df0068b2477          7987337.17           grocery    tra


Transactions per IST hour:
transaction_time
0     31081
1     31333
2     31052
3     31169
4     30998
5         0
6     52844
7     52140
8     52411
9     52812
10    51606
11    52576
12    52421
13    52291
14    53154
15    52215
16    52629
17    52778
18    52561
19    52188
20    52261
21    52531
22    52364
23    52585
Empty hours: [5]

Rows per split:
            transactions     labels  fraud_rate_%
split                                            
warmup          100000.0   100000.0         0.600
train           714745.0   714745.0         0.593
validation      144803.0   144803.0         0.641
holdout         140452.0   140452.0         0.595
total          1100000.0  1100000.0         0.600


## Step 3: Features, using only data from before each transaction
The trick that stops future leakage: sort each customer's transactions by time and only ever
look **backwards**. Rolling windows use `closed="left"`, which means the window is
`[t - window, t)`: it includes earlier rows and **excludes the current row and anything later**.

History includes every authorisation attempt (successful and failed), and the warmup split
(Oct to Dec 2024) is used as history for January 2025.

In [4]:
tx = transactions.copy()                                            # work on a copy, keep the raw table untouched
tx["ts"] = pd.to_datetime(tx.transaction_time, utc=True, format="ISO8601")            # text time -> real UTC timestamp
tx = tx.sort_values(["customer_id", "ts"]).reset_index(drop=True)   # each customer's history in time order

roll = tx.set_index("ts").groupby("customer_id")                    # per customer, indexed by time (needed for time windows)

# 1) amount_to_average_ratio = amount / customer's average amount over the preceding 90 days
avg_90d = roll["transaction_amount"].rolling("90D", closed="left").mean()  # mean of earlier attempts in [t-90d, t)
tx["avg_amount_90d"] = avg_90d.values                               # same row order as tx, so copy the values across
tx["amount_to_average_ratio"] = (tx.transaction_amount / tx.avg_amount_90d).round(4)  # NaN when there is no history

# 2) is_new_device = this device was not used by this customer in the preceding 90 days
prev_use = tx.groupby(["customer_id", "device_id"]).ts.shift(1)     # time this customer last used this same device
new_dev = prev_use.isna() | ((tx.ts - prev_use) > pd.Timedelta(days=NEW_DEVICE_DAYS))  # never used, or last use > 90 days ago
tx["is_new_device"] = new_dev.astype("boolean").where(tx.device_id.notna())  # unknown device -> stays null (no guess)


# 3) customer_merchant_distance_km = haversine distance between the two locations
def lat_lon(col):
    """Pull lat and lon out of a JSON location column like {"city":"Delhi","lat":28.61,"lon":77.21}."""
    parsed = col.map(lambda v: json.loads(v) if isinstance(v, str) else None)       # text -> dict (None if missing)
    lat = parsed.map(lambda d: d["lat"] if d else np.nan).astype(float)              # latitude, NaN if missing
    lon = parsed.map(lambda d: d["lon"] if d else np.nan).astype(float)              # longitude, NaN if missing
    city = parsed.map(lambda d: d["city"] if d else None)                            # city name, None if missing
    return lat, lon, city                                                            # three Series


def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two points on the earth."""
    p1, p2 = np.radians(lat1), np.radians(lat2)                     # latitudes in radians
    dphi = p2 - p1                                                  # difference in latitude
    dlmb = np.radians(lon2 - lon1)                                  # difference in longitude
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlmb / 2) ** 2  # haversine formula
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))              # central angle x earth radius = km


c_lat, c_lon, tx["customer_city"] = lat_lon(tx.customer_location)  # customer position at transaction time
m_lat, m_lon, _ = lat_lon(tx.merchant_location)                     # merchant registered position
tx["customer_merchant_distance_km"] = haversine_km(c_lat, c_lon, m_lat, m_lon).round(2)  # NaN when location is missing

# 4) is_night = transaction between 00:00 and 05:00 Indian time
tx["hour_ist"] = tx.ts.dt.tz_convert(IST).dt.hour                  # UTC -> IST, then take the hour (0-23)
tx["is_night"] = tx.hour_ist.between(NIGHT_START, NIGHT_END - 1)    # hours 0,1,2,3,4 are night (05:00 is not)

# 5) transaction_count_last_1_hour = earlier attempts by this customer in [t-1h, t)
cnt_1h = roll["amount_paise"].rolling("1h", closed="left").count()  # count rows in the window (current row excluded)
tx["transaction_count_last_1_hour"] = cnt_1h.fillna(0).astype(int).values  # empty window gives NaN -> 0

# 6) transaction_amount_last_24_hours = sum of earlier attempts by this customer in [t-24h, t), in paise
sum_24h = roll["amount_paise"].rolling("24h", closed="left").sum()  # add up the amounts in the window
tx["transaction_amount_last_24_hours"] = sum_24h.fillna(0).astype("int64").values  # empty window -> 0 paise

# Helpers needed by the reason codes in step 4 (same "before only" rule)
prev_city = tx.groupby(["customer_id", "customer_city"]).ts.shift(1)  # last time the customer was seen in this city
new_city = prev_city.isna() | ((tx.ts - prev_city) > pd.Timedelta(days=90))  # city not seen in the preceding 90 days
tx["is_new_city"] = new_city.astype("boolean").where(tx.customer_city.notna())  # unknown location -> null

out = outcomes[["transaction_id", "status", "outcome_time"]].copy()  # outcome of each attempt
out["outcome_ts"] = pd.to_datetime(out.outcome_time, utc=True, format="ISO8601")      # text -> timestamp
tx = tx.merge(out[["transaction_id", "status", "outcome_ts"]], on="transaction_id", how="left")  # attach outcome to each attempt
failed = tx[tx.status == "FAILED"].sort_values("outcome_ts")        # only failed attempts, ordered by when the failure was known
fail_times = failed.groupby("customer_id").outcome_ts.apply(lambda s: s.values)  # per customer: array of failure times
failed_24h = np.zeros(len(tx), dtype=int)                           # result: failures known in the 24h before each attempt
for cust, idx in tx.groupby("customer_id").indices.items():         # loop over customers (25,000, quick)
    if cust not in fail_times.index:                                # customer never failed ...
        continue                                                    # ... so all counts stay 0
    f = fail_times[cust]                                            # this customer's failure times (sorted)
    t = tx.ts.values[idx]                                           # this customer's attempt times
    hi = np.searchsorted(f, t, side="right")                        # failures whose outcome arrived at or before t
    lo = np.searchsorted(f, t - np.timedelta64(24, "h"), side="left")  # failures older than 24 h
    failed_24h[idx] = hi - lo                                       # failures known inside [t-24h, t]
tx["failed_last_24h"] = failed_24h                                  # store the count

cust_created = pd.to_datetime(customers.set_index("customer_id").created_at, utc=True, format="ISO8601")  # account creation time per customer
tx["account_age_days"] = ((tx.ts - tx.customer_id.map(cust_created)).dt.total_seconds() // 86400).astype(int)  # complete days

FEATURES = ["amount_to_average_ratio", "is_new_device", "customer_merchant_distance_km", "is_night",
            "transaction_count_last_1_hour", "transaction_amount_last_24_hours"]  # the six features of step 3
print(tx[["transaction_id", "customer_id", "transaction_time"] + FEATURES].head(10))  # first rows as a check
print(tx[FEATURES].describe(include="all").T)                       # quick summary of every feature

  transaction_id         customer_id               transaction_time  amount_to_average_ratio  is_new_device  customer_merchant_distance_km  is_night  transaction_count_last_1_hour  \
0  TX_0000194327  C_0003b241c8a23d2d  2025-02-10T02:56:37.280+00:00                      NaN           <NA>                        1358.16     False                              0   
1  TX_0000195527  C_0003b241c8a23d2d  2025-02-10T13:38:06.343+00:00                   0.1077           True                        1607.03     False                              0   
2  TX_0000263503  C_0003b241c8a23d2d  2025-03-11T16:43:29.256+00:00                   2.0150          False                           0.00     False                              0   
3  TX_0000268521  C_0003b241c8a23d2d  2025-03-14T06:30:55.578+00:00                   0.1993          False                        1532.79     False                              0   
4  TX_0000320326  C_0003b241c8a23d2d  2025-04-05T14:50:12.589+00:00                  

                                      count unique    top     freq           mean              std  min     25%     50%        75%          max
amount_to_average_ratio           1071675.0    NaN    NaN      NaN       2.573722       107.228818  0.0  0.4146  0.7484     1.3619   32317.9904
is_new_device                       1089317      2  False  1016899            NaN              NaN  NaN     NaN     NaN        NaN          NaN
customer_merchant_distance_km     1066933.0    NaN    NaN      NaN    1044.338982       555.435599  0.0  559.09  1081.9     1443.1      2428.81
is_night                            1100000      2  False   944367            NaN              NaN  NaN     NaN     NaN        NaN          NaN
transaction_count_last_1_hour     1100000.0    NaN    NaN      NaN       1.176318         1.883147  0.0     0.0     0.0        2.0         17.0
transaction_amount_last_24_hours  1100000.0    NaN    NaN      NaN  542252.070538  10815053.546141  0.0     0.0     0.0  258479.75  7998

## Step 4: Reason codes R01 to R08
| Code | Name | Rule (thresholds from class) |
|---|---|---|
| R01 | NEW_DEVICE | device not seen for this customer in the preceding 90 days |
| R02 | GEO_MISMATCH | new city in the preceding 90 days, or customer to merchant distance above 500 km |
| R03 | ODD_HOUR | between 00:00 and 05:00 IST |
| R04 | FAILED_ATTEMPTS | at least 3 failed attempts in the preceding 24 hours |
| R05 | AMOUNT_SPIKE | amount at least 5 times the 90-day average |
| R06 | NEW_ACCOUNT | account younger than 30 days |
| R07 | VELOCITY | at least 4 preceding attempts within 1 hour |
| R08 | HIGH_RISK_MCC | merchant category is gaming, gift_cards or electronics |

In [5]:
rules = {                                                           # code -> True/False for every transaction
    "R01_NEW_DEVICE": tx.is_new_device.fillna(False).astype(bool),  # unknown device does not fire
    "R02_GEO_MISMATCH": tx.is_new_city.fillna(False).astype(bool)   # new city ...
                        | (tx.customer_merchant_distance_km > GEO_DISTANCE_KM),  # ... or more than 500 km away
    "R03_ODD_HOUR": tx.is_night,                                    # night time in IST
    "R04_FAILED_ATTEMPTS": tx.failed_last_24h >= FAILED_MIN,        # 3+ failures in 24 h
    "R05_AMOUNT_SPIKE": tx.amount_to_average_ratio >= SPIKE_RATIO,  # 5x the 90-day average (NaN -> False)
    "R06_NEW_ACCOUNT": tx.account_age_days < NEW_ACCOUNT_DAYS,      # account younger than 30 days
    "R07_VELOCITY": tx.transaction_count_last_1_hour >= VELOCITY_MIN,  # 4+ attempts in the last hour
    "R08_HIGH_RISK_MCC": tx.merchant_category.isin(HIGH_RISK_MCC),  # risky merchant category
}
flags = pd.DataFrame(rules)                                         # one True/False column per code
tx["reason_codes"] = flags.apply(lambda r: [c for c in flags.columns if r[c]], axis=1)  # list of codes that fired, in R01..R08 order
print(flags.sum().rename("transactions flagged"))                   # how often each code fires
print(tx[["transaction_id", "reason_codes"]].head(10))              # a few examples

R01_NEW_DEVICE          72418
R02_GEO_MISMATCH       895465
R03_ODD_HOUR           155633
R04_FAILED_ATTEMPTS     98634
R05_AMOUNT_SPIKE        30690
R06_NEW_ACCOUNT         11541
R07_VELOCITY           148480
R08_HIGH_RISK_MCC      324094
Name: transactions flagged, dtype: int64
  transaction_id                                       reason_codes
0  TX_0000194327                [R02_GEO_MISMATCH, R06_NEW_ACCOUNT]
1  TX_0000195527  [R01_NEW_DEVICE, R02_GEO_MISMATCH, R06_NEW_ACC...
2  TX_0000263503                                                 []
3  TX_0000268521                                 [R02_GEO_MISMATCH]
4  TX_0000320326                                 [R02_GEO_MISMATCH]
5  TX_0000322856                                 [R02_GEO_MISMATCH]
6  TX_0000322859                                 [R02_GEO_MISMATCH]
7  TX_0000431762                                 [R02_GEO_MISMATCH]
8  TX_0000437018                                 [R02_GEO_MISMATCH]
9  TX_0000449897                       

## Step 5: Compare with the provided features table
A value matches when both are null, or both are present and equal
(decimals within 0.0001 for the ratio and 0.01 km for the distance, because the provided table is rounded).

In [6]:
given = features_given.rename(columns={                             # provided names -> my names
    "amount_to_avg_ratio": "amount_to_average_ratio",
    "txn_count_last_1h": "transaction_count_last_1_hour",
    "txn_amount_last_24h_paise": "transaction_amount_last_24_hours",
})
cmp = tx.merge(given, on="transaction_id", suffixes=("_mine", "_given"))  # one row per transaction, both versions side by side
TOLERANCE = {"amount_to_average_ratio": 1e-4, "customer_merchant_distance_km": 0.011}  # rounding tolerance for decimals


def to_bool(s):
    """Turn True/False/'true'/'false'/None into a nullable boolean column."""
    return s.map(lambda v: None if pd.isna(v) else str(v).lower() in ("true", "1")).astype("boolean")  # common format


def match_mask(feature, a, b):
    """True where my value and the given value agree (both null counts as agreement)."""
    both_null = a.isna() & b.isna()                                 # both sides empty
    if feature in TOLERANCE:                                        # decimal feature
        same = np.isclose(a.astype(float), b.astype(float), atol=TOLERANCE[feature])  # close enough after rounding
    elif feature in ("is_new_device", "is_night"):                  # boolean feature
        a, b = to_bool(a), to_bool(b)                               # same format on both sides
        same = (a == b).fillna(False).to_numpy(dtype=bool)          # equal where both present
    else:                                                           # integer feature
        same = (a.astype("Int64") == b.astype("Int64")).fillna(False).to_numpy(dtype=bool)  # exact integer match
    return both_null.to_numpy() | same                              # agreement


def compare(df, feature_list):
    """Match rate per feature, with a breakdown of the kinds of mismatch."""
    rows = []                                                       # one row per feature
    for f in feature_list:                                          # every feature
        a, b = df[f + "_mine"], df[f + "_given"]                    # my column and the provided column
        ok = match_mask(f, a, b)                                    # agreement per row
        rows.append({                                               # one line of the report
            "feature": f,                                           # feature name
            "compared": len(df),                                    # rows compared
            "matches": int(ok.sum()),                               # rows that agree
            "match_rate_%": round(100 * ok.mean(), 4),              # agreement as a percentage
            "mismatches": int((~ok).sum()),                         # rows that disagree
            "only_mine_null": int((a.isna() & b.notna()).sum()),    # I have no value, they do
            "only_given_null": int((a.notna() & b.isna()).sum()),   # they have no value, I do
            "both_null": int((a.isna() & b.isna()).sum()),          # both empty (counted as a match)
        })
    return pd.DataFrame(rows)                                       # report


match_report = compare(cmp, FEATURES)                               # compare the six features
print(match_report.to_string(index=False))                          # print it
match_report.to_csv(f"{OUTPUT_DIR}/step5_match_rate.csv", index=False)  # save it

by_split = pd.DataFrame({                                           # same match rates, per split
    f: cmp.assign(ok=match_mask(f, cmp[f + "_mine"], cmp[f + "_given"])).groupby("split").ok.mean() * 100  # % per split
    for f in FEATURES}).round(4)                                    # one column per feature
print(by_split)                                                     # print it
by_split.to_csv(f"{OUTPUT_DIR}/step5_match_rate_by_split.csv")     # save it

for f in FEATURES:                                                  # write every mismatching row to a file to inspect
    bad = cmp[~match_mask(f, cmp[f + "_mine"], cmp[f + "_given"])]  # rows that disagree on this feature
    if len(bad):                                                    # only when there is something to show
        bad[["transaction_id", "customer_id", "transaction_time", "split", f + "_mine", f + "_given"]].to_csv(
            f"{OUTPUT_DIR}/step5_mismatches_{f}.csv", index=False)  # one CSV per feature
        print(f"{f}: {len(bad):,} mismatches written to step5_mismatches_{f}.csv")  # tell the user

# Reason codes: compare the set of codes per transaction, and each code on its own
given_codes = cmp.reason_codes_given.map(lambda v: set(json.loads(v)) if isinstance(v, str) else set(v))  # provided codes as sets
mine_codes = cmp.reason_codes_mine.map(set)                         # my codes as sets
print(f"\nReason code sets identical: {100 * (given_codes == mine_codes).mean():.4f}% of transactions")  # whole-list match
code_rows = []                                                      # one row per code
for code in rules:                                                  # every reason code
    g = given_codes.map(lambda s: code in s)                        # did the provided table fire it?
    m = mine_codes.map(lambda s: code in s)                         # did my rule fire it?
    code_rows.append({"code": code, "fired_given": int(g.sum()), "fired_mine": int(m.sum()),  # how often each side fired
                      "match_rate_%": round(100 * (g == m).mean(), 4), "mismatches": int((g != m).sum())})  # agreement
code_report = pd.DataFrame(code_rows)                               # report
print(code_report.to_string(index=False))                           # print it
code_report.to_csv(f"{OUTPUT_DIR}/step4_reason_code_match.csv", index=False)  # save it

                         feature  compared  matches  match_rate_%  mismatches  only_mine_null  only_given_null  both_null
         amount_to_average_ratio   1100000  1100000         100.0           0               0                0      28325
                   is_new_device   1100000  1100000         100.0           0               0                0      10683
   customer_merchant_distance_km   1100000  1100000         100.0           0               0                0      33067
                        is_night   1100000  1100000         100.0           0               0                0          0
   transaction_count_last_1_hour   1100000  1100000         100.0           0               0                0          0
transaction_amount_last_24_hours   1100000  1100000         100.0           0               0                0          0


            amount_to_average_ratio  is_new_device  customer_merchant_distance_km  is_night  transaction_count_last_1_hour  transaction_amount_last_24_hours
split                                                                                                                                                       
holdout                       100.0          100.0                          100.0     100.0                          100.0                             100.0
train                         100.0          100.0                          100.0     100.0                          100.0                             100.0
validation                    100.0          100.0                          100.0     100.0                          100.0                             100.0
warmup                        100.0          100.0                          100.0     100.0                          100.0                             100.0



Reason code sets identical: 100.0000% of transactions


               code  fired_given  fired_mine  match_rate_%  mismatches
     R01_NEW_DEVICE        72418       72418         100.0           0
   R02_GEO_MISMATCH       895465      895465         100.0           0
       R03_ODD_HOUR       155633      155633         100.0           0
R04_FAILED_ATTEMPTS        98634       98634         100.0           0
   R05_AMOUNT_SPIKE        30690       30690         100.0           0
    R06_NEW_ACCOUNT        11541       11541         100.0           0
       R07_VELOCITY       148480      148480         100.0           0
  R08_HIGH_RISK_MCC       324094      324094         100.0           0


### Step 5b: Where mismatches come from
My six features match the provided table on every row. To show *why* each design choice matters,
this cell recomputes each feature with a common mistake and measures how far the match rate drops.
Every drop below is a mismatch explained.

In [7]:
def rate(feature, mine_values, df=cmp):
    """Match rate (%) of an alternative calculation against the provided column."""
    return round(100 * match_mask(feature, pd.Series(np.asarray(mine_values), index=df.index), df[feature + "_given"]).mean(), 2)  # % agreement


roll_c = cmp.set_index("ts").groupby("customer_id")                 # cmp is in the same order as tx (sorted by customer, time)
ok_only = cmp.amount_paise.where(cmp.status == "SUCCESS")           # amounts of successful attempts only (failed -> NaN)
roll_ok = ok_only.set_axis(cmp.ts).groupby(cmp.customer_id.values)  # per customer rolling over successful attempts only
mistakes = [                                                        # (feature, mistake, alternative values)
    ("amount_to_average_ratio", "include the current transaction in the 90-day average (leakage)",
     (cmp.transaction_amount / roll_c["transaction_amount"].rolling("90D", closed="both").mean().values).round(4)),
    ("amount_to_average_ratio", "30-day window instead of 90 days",
     (cmp.transaction_amount / roll_c["transaction_amount"].rolling("30D", closed="left").mean().values).round(4)),
    ("amount_to_average_ratio", "average over successful attempts only (drop failed ones)",
     (cmp.amount_paise / roll_ok.rolling("90D", closed="left").mean().values).round(4)),
    ("is_new_device", "device never seen before (no 90-day limit)",
     cmp.groupby(["customer_id", "device_id"]).ts.shift(1).isna().astype("boolean").where(cmp.device_id.notna())),
    ("is_new_device", "missing device treated as a new device",
     cmp.is_new_device_mine.fillna(True)),
    ("customer_merchant_distance_km", "equator radius 6,378 km instead of mean radius 6,371 km",
     (cmp.customer_merchant_distance_km_mine * 6378.137 / 6371.0).round(2)),
    ("customer_merchant_distance_km", "missing customer location filled with 0 km",
     cmp.customer_merchant_distance_km_mine.fillna(0)),
    ("is_night", "night hour taken in UTC instead of IST",
     cmp.ts.dt.hour < NIGHT_END),
    ("transaction_count_last_1_hour", "count includes the current transaction (leakage)",
     roll_c["amount_paise"].rolling("1h", closed="both").count().values),
    ("transaction_count_last_1_hour", "count successful attempts only",
     roll_ok.rolling("1h", closed="left").count().fillna(0).values),
    ("transaction_amount_last_24_hours", "sum includes the current transaction (leakage)",
     roll_c["amount_paise"].rolling("24h", closed="both").sum().values),
    ("transaction_amount_last_24_hours", "sum in rupees instead of paise",
     roll_c["transaction_amount"].rolling("24h", closed="left").sum().fillna(0).round().values),
]
mistake_report = pd.DataFrame(                                      # one row per mistake
    [{"feature": f, "mistake": why, "match_rate_%": rate(f, vals)} for f, why, vals in mistakes])  # measure each one
print(mistake_report.to_string(index=False))                        # print it
mistake_report.to_csv(f"{OUTPUT_DIR}/step5_common_mistakes.csv", index=False)  # save it

# Mistake that the task invites: computing on a small random sample of transactions WITHOUT their history
smp = transactions.sample(min(5000, len(transactions)), random_state=7).copy()  # 5,000 random rows, history not included
smp["ts"] = pd.to_datetime(smp.transaction_time, utc=True, format="ISO8601")          # text -> timestamp
smp = smp.sort_values(["customer_id", "ts"]).reset_index(drop=True)  # per customer in time order
sr = smp.set_index("ts").groupby("customer_id")                     # rolling inside the sample only
smp["amount_to_average_ratio_mine"] = (smp.transaction_amount / sr["transaction_amount"].rolling("90D", closed="left").mean().values).round(4)  # ratio
smp["transaction_count_last_1_hour_mine"] = sr["amount_paise"].rolling("1h", closed="left").count().fillna(0).values  # count
smp["transaction_amount_last_24_hours_mine"] = sr["amount_paise"].rolling("24h", closed="left").sum().fillna(0).values  # sum
smp = smp.merge(given[["transaction_id", "amount_to_average_ratio", "transaction_count_last_1_hour",
                       "transaction_amount_last_24_hours"]].add_suffix("_given")
                .rename(columns={"transaction_id_given": "transaction_id"}), on="transaction_id")  # attach provided values
sample_report = compare(smp, ["amount_to_average_ratio", "transaction_count_last_1_hour", "transaction_amount_last_24_hours"])  # compare
print("\nSample of 5,000 rows computed WITHOUT the customers' history:")  # heading
print(sample_report[["feature", "match_rate_%", "mismatches"]].to_string(index=False))  # print it
sample_report.to_csv(f"{OUTPUT_DIR}/step5_sample_without_history.csv", index=False)  # save it

# Edge case behind R04: a failure whose outcome arrives in the same millisecond as the next attempt
strict = np.zeros(len(tx), dtype=int)                               # failures counted with a strict "before" (<)
for cust, idx in tx.groupby("customer_id").indices.items():         # same loop as in step 3
    if cust in fail_times.index:                                    # only customers with failures
        f, t = fail_times[cust], tx.ts.values[idx]                  # failure times and attempt times
        strict[idx] = np.searchsorted(f, t, "left") - np.searchsorted(f, t - np.timedelta64(24, "h"), "left")  # outcome strictly before t
tie_rows = tx[strict != tx.failed_last_24h]                         # rows where "<" and "<=" disagree
print(f"\nAttempts where a failure arrived in the same millisecond: {len(tie_rows)}")  # how many
print(tie_rows[["transaction_id", "customer_id", "transaction_time", "failed_last_24h"]])  # show them

# Save my features and reason codes
tx["reason_codes"] = tx.reason_codes.map(json.dumps)               # list -> JSON text for the CSV
tx[["transaction_id", "customer_id", "transaction_time", "split"] + FEATURES + ["reason_codes"]].to_csv(
    f"{OUTPUT_DIR}/my_features.csv.gz", index=False)                # all rows, compressed
print(f"\nSaved results in {OUTPUT_DIR}/")                         # done

                         feature                                                         mistake  match_rate_%
         amount_to_average_ratio include the current transaction in the 90-day average (leakage)          0.20
         amount_to_average_ratio                                30-day window instead of 90 days         11.42
         amount_to_average_ratio        average over successful attempts only (drop failed ones)         63.03
                   is_new_device                      device never seen before (no 90-day limit)         99.14
                   is_new_device                          missing device treated as a new device         99.03
   customer_merchant_distance_km         equator radius 6,378 km instead of mean radius 6,371 km          9.15
   customer_merchant_distance_km                      missing customer location filled with 0 km         96.99
                        is_night                          night hour taken in UTC instead of IST         63.34
 


Sample of 5,000 rows computed WITHOUT the customers' history:
                         feature  match_rate_%  mismatches
         amount_to_average_ratio          2.74        4863
   transaction_count_last_1_hour         55.98        2201
transaction_amount_last_24_hours         53.10        2345



Attempts where a failure arrived in the same millisecond: 1
       transaction_id         customer_id               transaction_time  failed_last_24h
262003  TX_0000747401  C_3caecb0fb9610608  2025-10-03T19:06:28.035+00:00                3



Saved results in outputs/
